# TurboQuant × Llama-3.2: QJL Win-Rate Analysis

**Research question**: Does QJL's bias-correction benefit ever outweigh its added 1-bit projection variance?
- Phase 1 (GPT-2 Medium, d_head=64): **0/16 heads win**
- Phase 2 (Llama-3.2-1B, d_head=64): does QJL win at d=64 on a different model?
- Phase 2 (Llama-3.2-3B, d_head=128): does QJL flip at d=128?

**Repo**: https://github.com/G26karthik/kvquant-lab

## Setup
1. Add your HuggingFace token as a **Kaggle Secret** named `HF_TOKEN`.
   - OR use the **no-gating alternatives** in Cell 5/6 (SmolLM2 / Qwen2.5 — no license needed).
2. Enable GPU accelerator (T4 is sufficient).
3. Run all cells sequentially.

In [ ]:
# Cell 0: Install dependencies
import subprocess, sys
pkgs = [
    'transformers>=4.43.0',
    'huggingface_hub',
    'accelerate',
    'torch',
    'numpy',
    'matplotlib',
    'scipy',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
print('Dependencies installed.')

In [ ]:
# Cell 1: Clone the kvquant-lab repo (contains turboquant module)
import os, subprocess

REPO_URL = 'https://github.com/G26karthik/kvquant-lab.git'
REPO_DIR = '/kaggle/working/kvquant-lab'

if not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', REPO_URL, REPO_DIR])
    print(f'Cloned to {REPO_DIR}')
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull'])
    print(f'Updated repo at {REPO_DIR}')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('Repo on sys.path')

In [ ]:
# Cell 2: HuggingFace authentication (OPTIONAL — only needed for gated Llama models)
#
# If you are using the non-gated alternatives (SmolLM2 / Qwen2.5) in Cell 5/6,
# you can skip authentication entirely — just set HF_TOKEN = '' below.
#
# If you ARE using Llama 3.2, add HF_TOKEN as a Kaggle Secret.
import os

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('HuggingFace authentication successful.')
else:
    print('No HF_TOKEN found — proceeding without auth (fine for non-gated models).')

In [ ]:
# Cell 3: Global imports and shared constants
import json, math, os, sys, time
import torch
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

REPO_DIR = '/kaggle/working/kvquant-lab'
sys.path.insert(0, REPO_DIR)
from turboquant.turbo_quant_demo import TurboQuantMSE, TurboQuantProd

SEED   = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.float16

torch.manual_seed(SEED)
np.random.seed(SEED)
if DEVICE == 'cuda':
    torch.cuda.manual_seed_all(SEED)

OUT_BASE = '/kaggle/working/results'
os.makedirs(OUT_BASE, exist_ok=True)

# -------------------------------------------------------------------------
# MODEL SELECTION
# -------------------------------------------------------------------------
# Option A — Gated (requires HF token + Llama licence approval):
#   MODEL_A = 'meta-llama/Llama-3.2-1B-Instruct'   # d_head=64
#   MODEL_B = 'meta-llama/Llama-3.2-3B-Instruct'   # d_head=128
#
# Option B — Fully open, no token needed (RECOMMENDED if Llama not approved yet):
#   MODEL_A = 'HuggingFaceTB/SmolLM2-1.7B-Instruct'  # d_head=64, Apache 2.0, Llama arch
#   MODEL_B = 'Qwen/Qwen2.5-1.5B-Instruct'            # d_head=128, Apache 2.0
#
# Uncomment the pair you want to use:

# --- Option A: Llama (gated) ---
MODEL_A = 'meta-llama/Llama-3.2-1B-Instruct'
MODEL_B = 'meta-llama/Llama-3.2-3B-Instruct'
SUBDIR_A = 'llama_1b'
SUBDIR_B = 'llama_3b'

# --- Option B: Fully open alternatives (uncomment to use) ---
# MODEL_A  = 'HuggingFaceTB/SmolLM2-1.7B-Instruct'
# MODEL_B  = 'Qwen/Qwen2.5-1.5B-Instruct'
# SUBDIR_A = 'smollm2_1b'
# SUBDIR_B = 'qwen25_1p5b'

print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Model A: {MODEL_A}')
print(f'Model B: {MODEL_B}')

# 4 diverse 128+ token prompts
PROMPTS = [
    (
        "The human brain is the most complex organ in the known universe. It contains approximately 86 billion "
        "neurons, each forming thousands of synaptic connections with neighboring cells. This intricate network "
        "allows the brain to process sensory information, regulate bodily functions, store memories, and generate "
        "conscious thought. Modern neuroscience has made remarkable strides in understanding the architecture of "
        "the brain, but many fundamental questions remain unanswered. How do distributed neural circuits give "
        "rise to unified subjective experience? What is the precise mechanism of long-term memory consolidation? "
        "How does the brain distinguish relevant signals from irrelevant noise in real time? These questions drive "
        "ongoing research across cognitive science, computational neuroscience, and artificial intelligence. "
        "The parallels between biological neural networks and deep learning architectures continue to inspire "
        "new ideas in both fields."
    ),
    (
        "Climate change represents one of the most pressing challenges of the twenty-first century. Rising "
        "global temperatures, driven by the accumulation of greenhouse gases in the atmosphere, are already "
        "causing measurable shifts in weather patterns, sea levels, and biodiversity. The Intergovernmental "
        "Panel on Climate Change has repeatedly warned that limiting warming to 1.5 degrees Celsius above "
        "pre-industrial levels requires rapid, unprecedented reductions in carbon dioxide and methane emissions. "
        "Renewable energy technologies such as solar photovoltaics and wind turbines have achieved dramatic "
        "cost reductions over the past decade, making clean electricity increasingly competitive with fossil "
        "fuels. However, decarbonizing heavy industry, agriculture, and aviation presents formidable technical "
        "and economic obstacles. Carbon capture and storage, next-generation nuclear power, and green hydrogen "
        "are among the technologies being explored to fill gaps that electrification alone cannot bridge."
    ),
    (
        "The history of mathematics is a story of expanding abstraction. Ancient civilizations developed "
        "arithmetic for commerce and astronomy, while Greek mathematicians introduced the concept of proof "
        "and rigorous deduction. The invention of calculus by Newton and Leibniz in the seventeenth century "
        "provided the mathematical language for classical mechanics and much of physics. The nineteenth "
        "century saw an explosion of new structures: non-Euclidean geometry challenged the uniqueness of "
        "Euclid's axioms, abstract algebra generalized the properties of numbers to groups and rings, and "
        "set theory provided a unified foundation for all of mathematics. The twentieth century brought "
        "revolutionary results such as Godel's incompleteness theorems, which showed that any sufficiently "
        "powerful formal system contains true statements that cannot be proved within that system."
    ),
    (
        "Large language models have transformed natural language processing over the past several years. "
        "Unlike earlier recurrent or convolutional architectures, transformer-based models rely on the "
        "self-attention mechanism to model dependencies between all pairs of tokens in a sequence simultaneously. "
        "This architectural choice enables efficient parallelization during training and allows models to "
        "capture long-range relationships that earlier systems struggled with. Scaling laws suggest that "
        "model performance improves predictably with increases in both the number of parameters and the "
        "volume of training data. Efficient inference is a major practical concern: compressing the key-value "
        "cache through quantization, pruning, or low-rank approximation reduces memory bandwidth requirements "
        "and enables deployment of large models on resource-constrained hardware."
    ),
]

In [ ]:
# Cell 4: Core analysis function (works for ANY Llama-architecture model)

def run_winrate(hf_model_id: str, out_subdir: str):
    """Load a Llama-arch model, run 4-prompt QJL win-rate analysis, return summary dict."""
    out_dir = os.path.join(OUT_BASE, out_subdir)
    os.makedirs(out_dir, exist_ok=True)

    print(f'\n{"="*70}')
    print(f'  Loading {hf_model_id} ...')
    print(f'{"="*70}')

    tokenizer = AutoTokenizer.from_pretrained(hf_model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        hf_model_id,
        torch_dtype=DTYPE,
        device_map='auto',
    )
    model.eval()

    n_layer    = model.config.num_hidden_layers
    n_q_heads  = model.config.num_attention_heads
    n_kv_heads = getattr(model.config, 'num_key_value_heads', n_q_heads)
    d_head     = model.config.hidden_size // n_q_heads
    gqa_ratio  = max(1, n_q_heads // n_kv_heads)

    print(f'  Layers={n_layer}, Q-heads={n_q_heads}, KV-heads={n_kv_heads}, d_head={d_head}, GQA={gqa_ratio}:1')

    # Detect which attribute holds the layers (Llama vs Qwen2)
    if hasattr(model, 'model') and hasattr(model.model, 'layers'):
        layer_list = model.model.layers
    else:
        raise RuntimeError(f'Cannot find layer list in model {hf_model_id}')

    def run_prompt(prompt_text):
        q_caps, k_caps = {}, {}
        hooks = []
        for li, layer in enumerate(layer_list):
            attn = layer.self_attn
            def make_q(i):
                def fn(m, inp, out): q_caps[i] = out.detach().float().cpu()
                return fn
            def make_k(i):
                def fn(m, inp, out): k_caps[i] = out.detach().float().cpu()
                return fn
            hooks.append(attn.q_proj.register_forward_hook(make_q(li)))
            hooks.append(attn.k_proj.register_forward_hook(make_k(li)))

        ids = tokenizer.encode(prompt_text, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            model(ids)
        for h in hooks:
            h.remove()

        all_q, all_k = [], []
        for li in range(n_layer):
            seq = q_caps[li].shape[0]
            q_h = q_caps[li].view(seq, n_q_heads, d_head).permute(1, 0, 2)
            k_h = k_caps[li].view(seq, n_kv_heads, d_head).permute(1, 0, 2)
            all_q.append(q_h)
            all_k.append(k_h)
        return all_q, all_k

    per_q, per_k = [], []
    for pi, prompt in enumerate(PROMPTS):
        ntok = len(tokenizer.encode(prompt))
        print(f'  Prompt {pi+1}: {ntok} tokens')
        q_list, k_list = run_prompt(prompt)
        per_q.append(q_list)
        per_k.append(k_list)

    # Print Q/K L2 norms per layer
    print('\n  Layer  ||Q|| mean    ||K|| mean')
    for l in range(n_layer):
        qn = torch.cat([torch.norm(per_q[p][l].float(), dim=-1).reshape(-1) for p in range(len(PROMPTS))])
        kn = torch.cat([torch.norm(per_k[p][l].float(), dim=-1).reshape(-1) for p in range(len(PROMPTS))])
        print(f'  {l:>4}  {qn.mean().item():>10.4f}    {kn.mean().item():>10.4f}')

    # Quantizers
    mse4 = TurboQuantMSE(d_head, bits=4, device='cpu', seed=SEED)
    qjl4 = TurboQuantProd(d_head, bits=4, device='cpu', seed=SEED)

    n_p = len(PROMPTS)
    mse4_acc = np.zeros((n_layer, n_kv_heads, n_p))
    qjl4_acc = np.zeros((n_layer, n_kv_heads, n_p))

    for pi in range(n_p):
        for l in range(n_layer):
            k_all = per_k[pi][l].float()
            q_all = per_q[pi][l].float()
            seq   = k_all.shape[1]
            mask  = torch.tril(torch.ones(seq, seq, dtype=torch.bool))

            for kv_h in range(n_kv_heads):
                k = k_all[kv_h]
                q = q_all[(kv_h * gqa_ratio) % n_q_heads]

                q_hat4    = mse4.dequantize(*mse4.quantize(q)).float()
                k_hat4    = mse4.dequantize(*mse4.quantize(k)).float()
                q_hat_qjl = qjl4.dequantize(*qjl4.quantize(q)).float()
                k_hat_qjl = qjl4.dequantize(*qjl4.quantize(k)).float()

                true_ips = (q @ k.T)[mask]
                mse4_acc[l, kv_h, pi] = torch.mean(((q_hat4 @ k_hat4.T)[mask] - true_ips)**2).item()
                qjl4_acc[l, kv_h, pi] = torch.mean(((q_hat_qjl @ k_hat_qjl.T)[mask] - true_ips)**2).item()

    mse4_mean = mse4_acc.mean(axis=2)
    qjl4_mean = qjl4_acc.mean(axis=2)
    win_mat   = qjl4_mean < mse4_mean
    rel_imp   = np.where(mse4_mean > 0, (mse4_mean - qjl4_mean) / mse4_mean, 0.0)

    wins_per_layer = win_mat.sum(axis=1)
    wins_per_head  = win_mat.sum(axis=0)
    total_wins     = int(win_mat.sum())
    total_cells    = n_layer * n_kv_heads

    print(f'\n  === Win-Rate Summary: {hf_model_id} ===')
    print(f'  d_head={d_head}, KV-heads={n_kv_heads}, Layers={n_layer}')
    print(f'  QJL wins: {total_wins}/{total_cells} KV-head x layer cells')
    print(f'  Avg wins/layer: {wins_per_layer.mean():.2f}/{n_kv_heads}')

    layer_results = []
    for l in range(n_layer):
        heads = []
        for h in range(n_kv_heads):
            heads.append({
                'kv_head': h,
                'mse4': {'total_mse': float(mse4_mean[l, h])},
                'qjl4': {'total_mse': float(qjl4_mean[l, h])},
                'qjl_better': bool(win_mat[l, h]),
                'relative_improvement': float(rel_imp[l, h]),
            })
        layer_results.append({'layer': l, 'heads': heads})

    result = {
        'model': hf_model_id, 'd_head': d_head,
        'n_layer': n_layer, 'n_q_heads': n_q_heads, 'n_kv_heads': n_kv_heads,
        'total_wins': total_wins, 'total_cells': total_cells,
        'wins_per_layer': wins_per_layer.tolist(),
        'wins_per_head': wins_per_head.tolist(),
        'layer_results': layer_results,
    }
    json_path = os.path.join(out_dir, 'winrate_data.json')
    with open(json_path, 'w') as f:
        json.dump(result, f, indent=2)
    print(f'  JSON saved: {json_path}')

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(12, max(4, n_layer // 4)))
    im = axes[0].imshow(rel_imp, aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
    axes[0].set_title(f'QJL vs MSE Relative Improvement\n{hf_model_id.split("/")[-1]} (d={d_head})', fontsize=10)
    axes[0].set_xlabel('KV Head'); axes[0].set_ylabel('Layer')
    fig.colorbar(im, ax=axes[0])
    axes[1].bar(range(n_kv_heads), wins_per_head, color='#4a9eff', edgecolor='black', alpha=0.8)
    axes[1].axhline(y=n_layer/2, color='red', linestyle='--', label=f'50% ({n_layer//2} layers)')
    axes[1].set_title(f'Layers where QJL Wins\n{hf_model_id.split("/")[-1]} (d={d_head})', fontsize=10)
    axes[1].set_xlabel('KV Head'); axes[1].set_ylabel('Win Count')
    axes[1].set_ylim(0, n_layer + 1); axes[1].legend()
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, 'winrate_plot.png'), dpi=120)
    plt.show(); plt.close()

    del model
    import gc; gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

    return result

print('Analysis function defined. Ready.')

In [ ]:
# Cell 5: Run Model A (d_head=64)
# Default: Llama-3.2-1B | Fallback: SmolLM2-1.7B (change MODEL_A in Cell 3)
result_a = run_winrate(hf_model_id=MODEL_A, out_subdir=SUBDIR_A)

In [ ]:
# Cell 6: Run Model B (d_head=128)
# Default: Llama-3.2-3B | Fallback: Qwen2.5-1.5B (change MODEL_B in Cell 3)
result_b = run_winrate(hf_model_id=MODEL_B, out_subdir=SUBDIR_B)

In [ ]:
# Cell 7: NN Recall at d=64 and d=128 (purely synthetic, no model needed)
from turboquant.turbo_quant_demo import TurboQuantProd as OrigTQP
from turboquant.wht_quantizer import AdaptiveTurboQuantProd
from turboquant.outlier_channel_quantizer import OutlierChannelQuantizer

def evaluate_nn(db, queries, true_top1, wrapper):
    chunk = 1000
    n_db, n_q = db.shape[0], queries.shape[0]
    compressed   = [wrapper.quantize(db[i:i+chunk]) for i in range(0, n_db, chunk)]
    recon_db     = torch.cat([wrapper.dequantize(c).float() for c in compressed], dim=0)
    compressed_q = [wrapper.quantize(queries[i:i+chunk]) for i in range(0, n_q, chunk)]
    recon_q      = torch.cat([wrapper.dequantize(c).float() for c in compressed_q], dim=0)
    approx_ip    = torch.matmul(recon_q, recon_db.T)
    approx_ranks = torch.argsort(approx_ip, dim=-1, descending=True)
    recall_1     = ((approx_ranks[:, 0] == true_top1).float().mean() * 100).item()
    recall_10    = sum(1 for qi in range(n_q) if true_top1[qi].item() in approx_ranks[qi, :10]) / n_q * 100
    cr           = (n_db * db.shape[1] * 16) / (n_db * wrapper.bits_per_vector())
    return {'recall_1': recall_1, 'recall_10': recall_10, 'compression_ratio': cr}

def run_nn_search(dim, out_subdir):
    print(f'\nNN Search at dim={dim}')
    torch.manual_seed(SEED)
    G  = torch.randn(10_000, dim); db = (G / (G.norm(dim=-1, keepdim=True) + 1e-8)).to(torch.float16)
    torch.manual_seed(1)
    Gq = torch.randn(500, dim);   queries = (Gq / (Gq.norm(dim=-1, keepdim=True) + 1e-8)).to(torch.float16)
    true_top1 = torch.argsort(torch.matmul(queries.float(), db.float().T), dim=-1, descending=True)[:, 0]
    results = {'flat_turboquant': {}, 'wht_asym': {}, 'outlier_split': {}}
    for b in [2, 3, 4]:
        class FW:
            def __init__(s): s.q=OrigTQP(dim, b, 'cpu', seed=SEED); s.dim=dim; s.bits=b
            def quantize(s,x): return s.q.quantize(x)
            def dequantize(s,c): return s.q.dequantize(*c)
            def bits_per_vector(s): return s.dim*(s.bits-1)+s.dim+32+32
        res = evaluate_nn(db, queries, true_top1, FW())
        results['flat_turboquant'][str(b)] = res
        print(f'  Flat {b}-bit: R@1={res["recall_1"]:.1f}%  CR={res["compression_ratio"]:.2f}x')
        class WA:
            def __init__(s): s.q=AdaptiveTurboQuantProd(dim,b,'cpu',seed=SEED); s.dim=dim; s.bits=b
            def quantize(s,x): return s.q.quantize(x)
            def dequantize(s,c): return s.q.dequantize(*c)
            def bits_per_vector(s): return s.dim*(s.bits-1)+s.dim+32+32 if s.q.use_qjl else s.dim*s.bits+32
        res2 = evaluate_nn(db, queries, true_top1, WA())
        results['wht_asym'][str(b)] = res2
        print(f'  WHT  {b}-bit: R@1={res2["recall_1"]:.1f}%  CR={res2["compression_ratio"]:.2f}x')
    for b_avg in [2.5, 3.5]:
        class OS:
            def __init__(s): s.q=OutlierChannelQuantizer(dim,b_avg,'cpu',seed=SEED); s.q.calibrate(db[:1000]); s.dim=dim
            def quantize(s,x): return s.q.quantize(x)
            def dequantize(s,c): return s.q.dequantize(c)
            def bits_per_vector(s): o=s.q.outlier_count; return o*s.q.outlier_bits+(s.dim-o)*s.q.normal_bits+32
        res3 = evaluate_nn(db, queries, true_top1, OS())
        results['outlier_split'][str(b_avg)] = res3
        print(f'  Outlier {b_avg}-bit: R@1={res3["recall_1"]:.1f}%  CR={res3["compression_ratio"]:.2f}x')
    out_dir = os.path.join(OUT_BASE, out_subdir)
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, 'nn_search.json'), 'w') as f:
        json.dump(results, f, indent=2)
    print(f'  Saved -> {out_dir}/nn_search.json')
    return results

nn_a = run_nn_search(dim=result_a['d_head'], out_subdir=SUBDIR_A)
nn_b = run_nn_search(dim=result_b['d_head'], out_subdir=SUBDIR_B)

In [ ]:
# Cell 8: Final Summary Table — the key crossover comparison
gpt2_json = os.path.join(REPO_DIR, 'results', 'theory', 'crossover_data_multiprompt.json')
gpt2_wins = gpt2_total = 0
if os.path.exists(gpt2_json):
    with open(gpt2_json) as f: g = json.load(f)
    gpt2_wins  = g.get('total_wins', 0)
    gpt2_total = g.get('total_cells', 384)

print('\n' + '='*70)
print('  QJL vs 4-bit MSE-only  —  Final Comparison')
print('  Repo: https://github.com/G26karthik/kvquant-lab')
print('='*70)
print(f'  {"Model":<40} {"d_head":>6}  {"QJL Wins":>10}  {"Total":>8}')
print('-'*70)
if gpt2_total:
    print(f'  {"GPT-2 Medium (Phase 1)":<40} {"64":>6}  {gpt2_wins:>10}  {gpt2_total:>8}')
ra = result_a; rb = result_b
print(f'  {ra["model"]:<40} {ra["d_head"]:>6}  {ra["total_wins"]:>10}  {ra["total_cells"]:>8}')
print(f'  {rb["model"]:<40} {rb["d_head"]:>6}  {rb["total_wins"]:>10}  {rb["total_cells"]:>8}')
print('='*70)
print(f'\n  Model A (d={ra["d_head"]}): {ra["total_wins"]}/{ra["total_cells"]} KV-head x layer cells win')
print(f'  Model B (d={rb["d_head"]}): {rb["total_wins"]}/{rb["total_cells"]} KV-head x layer cells win')
print()
if rb['total_wins'] > ra['total_wins']:
    print('  >> QJL wins MORE at d=128 than d=64 — crossover observed!')
elif rb['total_wins'] == ra['total_wins']:
    print('  >> Equal win-rate at d=64 and d=128 — no clear crossover.')
else:
    print('  >> QJL wins FEWER at d=128 than d=64 — no crossover.')
print('\nAll results: /kaggle/working/results/')

In [ ]:
# Cell 9: Package results for download
import shutil
archive_path = '/kaggle/working/turboquant_results'
shutil.make_archive(archive_path, 'zip', OUT_BASE)
print(f'Archive: {archive_path}.zip — download from the Kaggle output panel.')